# ED-SIM-02 Lab Tour

This notebook walks through a complete ED-SIM-02 simulation:
1. Load a predefined scenario
2. Run the simulation with full invariant tracking
3. Plot the core invariants over time
4. Inspect morphology evolution
5. Print a compact summary

The scenario is `A_2d_cosine`: a 2D isotropic cosine perturbation
around the equilibrium density, relaxing under the canonical ED PDE.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from edsim.experiments.scenarios import get_scenario
from edsim.experiments.atlas import run_atlas, summarize_time_series

In [ ]:
# Load and run the canonical 2D scenario
scenario = get_scenario("A_2d_cosine")
print(f"Scenario: {scenario.name}")
print(f"Description: {scenario.description}")

params, ts = run_atlas(scenario)
print(f"\nDimension: {params.d}")
print(f"Grid: {params.N}")
print(f"Time: T={params.T}, dt={params.dt}, snapshots={len(ts.times)}")

In [ ]:
# Plot core invariants: energy, complexity, spectral entropy
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].semilogy(ts.times, ts.energy, 'b-', linewidth=1.5)
axes[0].set_xlabel('t')
axes[0].set_ylabel('E[rho]')
axes[0].set_title('Lyapunov Energy (Law 2: monotone)')
axes[0].grid(True, alpha=0.3)

axes[1].semilogy(ts.times, ts.complexity, 'r-', linewidth=1.5)
axes[1].set_xlabel('t')
axes[1].set_ylabel('C[rho]')
axes[1].set_title('ED-Complexity')
axes[1].grid(True, alpha=0.3)

axes[2].plot(ts.times, ts.spectral_entropy, 'g-', linewidth=1.5)
axes[2].set_xlabel('t')
axes[2].set_ylabel('H')
axes[2].set_title('Spectral Entropy (Law 3: concentration)')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Plot morphology fractions over time
blobs = [m['blob'] for m in ts.morphology_fractions]
sheets = [m['sheet'] for m in ts.morphology_fractions]

fig, ax = plt.subplots(figsize=(8, 4))
ax.stackplot(ts.times, blobs, sheets,
             labels=['blob', 'sheet'], colors=['#2196F3', '#FF9800'], alpha=0.8)
ax.set_xlabel('t')
ax.set_ylabel('volume fraction')
ax.set_title('Morphology Evolution (2D: blob + sheet = 1)')
ax.legend(loc='center right')
ax.set_ylim(0, 1.05)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Plot dissipation channels and correlation length
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(ts.times, ts.R_grad, label='R_grad')
axes[0].plot(ts.times, ts.R_pen, label='R_pen')
axes[0].plot(ts.times, ts.R_part, label='R_part')
axes[0].set_xlabel('t')
axes[0].set_ylabel('fraction')
axes[0].set_title('Dissipation Channels (Law 5)')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(ts.times, ts.correlation_length, 'm-', linewidth=1.5)
axes[1].set_xlabel('t')
axes[1].set_ylabel('xi')
axes[1].set_title('Correlation Length (grows as high-k decays)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Print compact summary
summary = summarize_time_series(ts)

print("=" * 50)
print("  ED-SIM-02 Simulation Summary")
print("=" * 50)
print(f"T = {summary['T']:.2f}, snapshots = {summary['n_snapshots']}")
print()
print(f"Energy:     {summary['energy_initial']:.4e} -> {summary['energy_final']:.4e}  (ratio={summary['energy_ratio']:.4f})")
print(f"Complexity: {summary['complexity_initial']:.4e} -> {summary['complexity_final']:.4e}  (ratio={summary['complexity_ratio']:.4f})")
print(f"Mass:       {summary['mass_initial']:.6f} -> {summary['mass_final']:.6f}  (change={summary['mass_change']:.2e})")
print(f"H_spec:     {summary['spectral_entropy_initial']:.3f} -> {summary['spectral_entropy_final']:.3f}")
print(f"R_grad:     {summary['R_grad_mean']:.4f} (mean)")
print(f"xi:         {summary['correlation_length_initial']:.4f} -> {summary['correlation_length_final']:.4f}  (ratio={summary['xi_ratio']:.2f})")
print(f"chi:        {summary['euler_chi_initial']} -> {summary['euler_chi_final']}  (constant={summary['chi_constant']})")
print(f"v_final:    {summary['v_final']:+.4e}")
print(f"E monotone: {summary['energy_monotone']}")

## Interpretation

The results demonstrate the core ED architectural laws:

- **Law 1 (Unique Attractor):** The density field relaxes monotonically toward
  the equilibrium rho_star = 0.5, and the participation variable v decays toward 0.

- **Law 2 (Energy Monotonicity):** The Lyapunov energy E[rho] decreases at every
  time step, confirming the system is a gradient flow.

- **Law 3 (Spectral Concentration):** The spectral entropy H decreases over time,
  meaning the field concentrates into fewer modes as high-k components decay
  exponentially faster than low-k ones.

- **Morphology:** In 2D, the field is classified into blobs (local maxima/minima
  with both Hessian eigenvalues negative) and sheets (saddle regions). The blob
  fraction is stable around 15-20% throughout the relaxation.

- **Dissipation Channels:** The penalty channel R_pen dominates the gradient
  channel R_grad, consistent with the moderate-amplitude regime. The participation
  channel R_part is small (< 1%), confirming that v provides a gentle global
  feedback rather than a dominant forcing.

- **Correlation Length:** xi grows over time because the high-frequency spatial
  structure decays faster than the low-frequency structure, leaving the field
  increasingly dominated by long-wavelength modes.